<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "626"
iso_country = "Timor-Leste"
year = 2023
import_portfolio = "h5_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 2.080*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Timor-Leste': 

Number of trade records present in dataset: 14800
Number of partner countries present in dataset: 108
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$717.3m
Total weight within the dataset: 1346.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,626,Timor-Leste,036,Australia,Import,Sea,401691,401691 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",8651.37367,NaN,8651.37367,kg,617.64663,617.64663,840.913521,6044.0,3755.566324,3158.106500,['Container'],['Container'],9.713035,9.694244,9.696112,9.696112,9.705293,9.709447,10.069166,16.792709,10.738267,10.116559,10.149052,10.467243,9.886774,16.296286,11.239999,10.529058,10.329609,10.740021,9.727701,17.982279,13.480100,12.279411,11.499965,11.133092,9.571868,19.461406,15.704191,14.767779,12.146079,11.833291,9.438089,19.824802,18.748924,19.287782,13.086596,12.377915,9.249006,18.948536,18.362578,19.540608,15.465048,12.535247,9.097473,17.213644,17.773868,19.396065,15.155540,12.672394,8.943965,15.794719,16.186207,18.703189,15.431791,12.768696,3627
1,1,626,Timor-Leste,036,Australia,Import,Sea,401699,401699 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",6363.26126,NaN,6363.26126,kg,1733.62570,1733.62570,618.508994,6044.0,3755.566324,2322.851551,['Container'],['Container'],7.144135,7.130314,7.131687,7.131687,7.138440,7.141496,7.406076,12.351379,7.898214,7.440935,7.464834,7.698870,7.271923,11.986249,8.267248,7.744337,7.597637,7.899504,7.154922,13.226332,9.914888,9.031757,8.458458,8.188615,7.040304,14.314260,11.550752,10.862001,8.933688,8.703626,6.941907,14.581545,13.790215,14.186556,9.625457,9.104208,6.802832,13.937033,13.506049,14.372514,11.374857,9.219929,6.691377,12.660985,13.073041,14.266200,11.147208,9.320804,6.578468,11.617337,11.905284,13.756576,11.350396,9.391636,3627


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $215.6m (30.1%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
10 | Cereals               73164963.2  127327488.8     10.2      9.5
19 | Preparations of cere  27417408.3   11948618.7      3.8      0.9
02 | Meat and edible meat  18712480.6    9807794.7      2.6      0.7
22 | Beverages, spirits a  15128188.0  431003079.2      2.1     32.0
21 | Miscellaneous edible  12921939.2    3482785.9      1.8      0.3
15 | Animal, vegetable or  10703029.7   11746504.3      1.5      0.9
16 | Meat, fish, crustace   9551177.5    2178813.1      1.3      0.2
20 | Preparations of vege   8551355.4    6584070.9      1.2      0.5
04 | Dairy produce; birds   7780016.5    4578689.0      1.1      0.3
11 | Products of the mill   7511349.1   17684550.7      1.0      1.3
17 | Sugars and sugar con   6692959.9   12064357.3      0.9      0.9
23

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $206.4m (28.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       121180468.6  177804632.8     16.9     13.2
Food & Beverages   84756596.8  466323948.1     11.8     34.6
Fishing              455373.3      66937.3      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $186.8m (26.0%), ... 

                                 cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
PCR - Processed rice           73051414.5  126890307.0     10.2      9.4
OFD - Food products n.e.c.     60533266.8   39587679.9      8.4      2.9
OMT - Meat products n.e.c.     25158669.4   11306951.5      3.5      0.8
VOL - Vegetable oils and fats  10698873.7   11826097.4      1.5      0.9
MIL - Dairy products            6046836.8    2990445.3      0.8      0.2
V_F - Vegetables, fruit, nuts   5522480.5    8362588.5      0.8      0.6
SGR - Sugar                     4833964.2   10604632.0      0.7      0.8
CMT - Bovine meat prods          448188.7      96477.8      0.1      0.0
OCR - Crops n.e.c.               189576.0     138062.7      0.0      0.0
OAP - Animal products n.e.c.     156940.6      38789.4      0.0      0.0
GRO - Cereal grains n.e.c.       102586.8     411387.8    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 30.1% 
GTAP 26.0% 
EORA 28.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27855/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,626,Timor-Leste,036,Australia,Import,Sea,401691,401691 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",8651.37367,NaN,8651.37367,kg,617.64663,617.64663,840.913521,6044.0,3755.566324,3158.106500,['Container'],['Container'],9.713035,9.694244,9.696112,9.696112,9.705293,9.709447,10.069166,16.792709,10.738267,10.116559,10.149052,10.467243,9.886774,16.296286,11.239999,10.529058,10.329609,10.740021,9.727701,17.982279,13.480100,12.279411,11.499965,11.133092,9.571868,19.461406,15.704191,14.767779,12.146079,11.833291,9.438089,19.824802,18.748924,19.287782,13.086596,12.377915,9.249006,18.948536,18.362578,19.540608,15.465048,12.535247,9.097473,17.213644,17.773868,19.396065,15.155540,12.672394,8.943965,15.794719,16.186207,18.703189,15.431791,12.768696,3627,NaN
1,1,626,Timor-Leste,036,Australia,Import,Sea,401699,401699 | Rubber; vulcanised (,4016,4016 | Articles of vulcanis,40,40 | Rubber and articles,3627,Articles of vulcanised rubber n.e.c.; hard rub...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",6363.26126,NaN,6363.26126,kg,1733.62570,1733.62570,618.508994,6044.0,3755.566324,2322.851551,['Container'],['Container'],7.144135,7.130314,7.131687,7.131687,7.138440,7.141496,7.406076,12.351379,7.898214,7.440935,7.464834,7.698870,7.271923,11.986249,8.267248,7.744337,7.597637,7.899504,7.154922,13.226332,9.914888,9.031757,8.458458,8.188615,7.040304,14.314260,11.550752,10.862001,8.933688,8.703626,6.941907,14.581545,13.790215,14.186556,9.625457,9.104208,6.802832,13.937033,13.506049,14.372514,11.374857,9.219929,6.691377,12.660985,13.073041,14.266200,11.147208,9.320804,6.578468,11.617337,11.905284,13.756576,11.350396,9.391636,3627,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $158.0m of $717.3m (22.0%)
Volume: 198.3 of 1346.9 million kilograms (14.7%)
Value-to-Weight: 0.8 from 0.53 US$ per kg (+49.6%)
Average distance to market (volume-weighted): 3622.0 from 2011.4 nm (+80.1%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    504  114084000.8  157966236.1     15.9     11.7
1        Meat and Prep     61   18239184.2    9707671.4      2.5      0.7
2           Vegetables    250    9966075.1   11843292.7      1.4      0.9
3      Sugar and Honey     67    6694783.4   12064880.1      0.9      0.9
4                Dairy    141    4773675.1    2900431.9      0.7      0.2
5               Fruits    253    2818052.4    2669240.9      0.4      0.2
6        Fats and Oils     71    1353061.2     748406.8      0.2      0.1
7  Unprocessed Cereals     27     106695.6     425278.1      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,2473825.9,4125688.0,2638212.8,2485469.7,2493452.5,2571626.7,2318780.9,4870622.7,4606297.5,4738686.0,3215158.0,3041046.9,2197382.9,3880498.7,3976680.7,4595061.2,3791333.4,3137055.4
1,Fats and Oils,729486.5,1216590.7,777961.2,732920.0,735274.0,758326.1,683766.5,1436258.4,1358313.7,1397352.7,948091.9,896749.6,647968.4,1144288.8,1172651.1,1355000.3,1117995.5,925060.8
2,Fruits,3308849.0,5518285.8,3528723.5,3324422.9,3335100.4,3439661.7,3101469.5,6514668.0,6161121.6,6338197.0,4300412.6,4067531.4,2939094.5,5190334.5,5318982.1,6146092.7,5071072.0,4195947.0
3,Meat and Prep,24067529.7,40138280.4,25666828.2,24180810.0,24258474.3,25019020.2,22559116.7,47385652.2,44814066.7,46102057.6,31279852.9,29585948.0,21378051.6,37752865.2,38688607.8,44704750.8,36885387.3,30520002.9
4,Processed Cereals,172137194.7,287079358.5,183575791.0,172947404.5,173502880.1,178942501.1,161348635.1,338914435.9,320521790.9,329733835.2,223721595.3,211606349.5,152901352.1,270018252.9,276710925.3,319739935.4,263813826.3,218286951.0
5,Sugar and Honey,14094071.0,23505186.5,15030628.5,14160408.5,14205889.2,14651268.8,13210736.5,27749285.3,26243351.3,26997605.5,18317645.1,17325685.6,12519098.6,22108274.9,22656250.6,26179335.4,21600275.3,17872672.9
6,Unprocessed Cereals,350360.3,584308.4,373641.9,352009.3,353139.9,364211.5,328401.7,689811.1,652375.6,671125.4,455352.8,430694.0,311208.5,549583.0,563204.9,650784.2,536954.8,444291.4
7,Vegetables,11868315.9,19793215.0,12656971.0,11924177.3,11962475.6,12337520.2,11124478.8,23367079.9,22098965.1,22734106.5,15424897.4,14589589.4,10542065.3,18616905.7,19078344.3,22045058.6,18189130.0,15050195.8


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.35,0.03,0.00,0.00,0.02,0.53,0.48,0.51,0.19,0.15,0.35,0.37,0.50,0.33,0.20
1,Fats and Oils,0.36,0.04,0.00,0.00,0.02,0.56,0.50,0.53,0.20,0.16,0.37,0.39,0.52,0.35,0.20
2,Fruits,0.78,0.08,0.01,0.01,0.05,1.21,1.09,1.15,0.43,0.34,0.80,0.84,1.14,0.76,0.45
3,Meat and Prep,0.88,0.09,0.01,0.01,0.05,1.36,1.22,1.29,0.48,0.39,0.90,0.95,1.28,0.85,0.50
4,Processed Cereals,1.01,0.10,0.01,0.01,0.06,1.56,1.40,1.48,0.55,0.44,1.03,1.09,1.46,0.97,0.57
5,Sugar and Honey,1.41,0.14,0.01,0.02,0.08,2.17,1.95,2.06,0.76,0.61,1.43,1.51,2.04,1.36,0.80
6,Unprocessed Cereals,2.19,0.22,0.02,0.03,0.13,3.39,3.04,3.21,1.19,0.96,2.23,2.36,3.18,2.12,1.25
7,Vegetables,0.80,0.08,0.01,0.01,0.05,1.23,1.10,1.16,0.43,0.35,0.81,0.86,1.15,0.77,0.45
